In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
## Load the dataset
data = pd.read_csv('Churn_Modelling.csv')

In [33]:
data.head(1)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1


In [34]:
## Preprocess the dataset
# Drop unnecessary columns
data2= data.drop(columns=['RowNumber', 'CustomerId', 'Surname'], axis=1)
data2.head(1)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.0,1,1,1,101348.88,1


In [35]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
data2['Gender']= label_encoder_gender.fit_transform(data2['Gender'])

In [36]:
## One Hot Encoding for Geography
onehot_encoder_geo = OneHotEncoder()
geo_encoder= onehot_encoder_geo.fit_transform(data2[['Geography']])  
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [37]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [38]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head(2)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0


In [39]:
## Combine the one hot encoded columns with the original dataframe.
data3= pd.concat([data2.drop(columns=['Geography'], axis=1), geo_encoded_df], axis=1)
data3.head(2)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0


In [40]:
## Save encoder annd scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [41]:
data3.head(2)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0


In [42]:
## Divide dataset into independent and dependent features.
X = data3.drop('Exited',axis=1)
y = data3['Exited']

# Split the dataset into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scailing the features.
scaler = StandardScaler()
X_train_transformed = scaler.fit_transform(X_train)
X_test_transformed = scaler.transform(X_test)

In [43]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN Implementation

Note: In TensorFlow, the forward and backward propogation structure of ANN is known as a Sequential Model/Network.
Creating ANN in TensorFlow:
1. First initialize the Sequential N/W.
2. Dense Layer (Hidden Layer) (Dense 64 implies hidden layer with 64 neuron)
3. Activation Function --> Sigmoid, tanh, ReLu, Leaky ReLu
4. Optimizer --> Useful in Back Propogation (Helps in updating weights).
5. Define the Loss Function (Objective to minimize it).
6. Metrics --> 
7. TensorBoard --> Visualize the logs of the Training data.


In [47]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [48]:
## Build our Model.
model = Sequential(
[
    Dense(64, activation = 'relu',input_shape = (X_train_transformed.shape[1],)), #Hidden Layer 1 Connected with Input Layer.
    Dense(32, activation = 'relu'), #Hidden Layer 2
    Dense(1, activation = 'sigmoid') # Output Layer
]
)

In [49]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [51]:
## Defining the Optimizer
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy()

In [52]:
## Compile the model to perform forward and backward propogation.
#Either define as text
# model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])
#Or Use the customized ones defined above.
model.compile(optimizer=opt,loss=loss_fn,metrics=['accuracy'])

In [55]:
## Set up the TensorBoard.
log_dir_ = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir= log_dir_, histogram_freq=1)

Early Stopping: If the Loss decrease is less than certain threshold then stop the iteration, saves time required to run the rest of the Epochs.

In [59]:
## Setup Early Stopping
early_stopping_callback =EarlyStopping(monitor= 'val_loss',patience= 10,
                                       restore_best_weights=True)

In [60]:
## Train the model.

history = model.fit(
    X_train_transformed,y_train,
    validation_data=(X_test_transformed,y_test),
    epochs=100,
    callbacks= [tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3401 - accuracy: 0.8620 - val_loss: 0.3356 - val_accuracy: 0.8610
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3354 - accuracy: 0.8615 - val_loss: 0.3386 - val_accuracy: 0.8680
Epoch 3/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3317 - accuracy: 0.8643 - val_loss: 0.3523 - val_accuracy: 0.8580
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3339 - accuracy: 0.8631 - val_loss: 0.3367 - val_accuracy: 0.8610
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3269 - accuracy: 0.8655 - val_loss: 0.3398 - val_accuracy: 0.8600
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3256 - accuracy: 0.8681 - val_loss: 0.3407 - val_accuracy: 0.8615
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3262 - accuracy: 0.8651 - val_loss: 0.3663 - val_accuracy: 0.8585

In [62]:
## Saving the model
model.save('model.h5')

In [67]:
## Load TensorFlow Extension.
%reload_ext tensorboard

In [69]:
%tensorboard --logdir logs/fit20250427-173401

Reusing TensorBoard on port 6007 (pid 27268), started 0:02:57 ago. (Use '!kill 27268' to kill it.)

In [ ]:
# Killing the TensorBoard to save RAM
# !taskkill /PID 27268 /F

SUCCESS: The process with PID 27268 has been terminated.


In [72]:
{i:'' for i in data3.columns}

{'CreditScore': '',
 'Gender': '',
 'Age': '',
 'Tenure': '',
 'Balance': '',
 'NumOfProducts': '',
 'HasCrCard': '',
 'IsActiveMember': '',
 'EstimatedSalary': '',
 'Exited': '',
 'Geography_France': '',
 'Geography_Germany': '',
 'Geography_Spain': ''}